## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code, data, and pre-computed structural-feature files needed into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.
- Avoid `git clone`, which can be less reliable in hosted notebook runtimes.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/ss3/deepalgpro_train_ss3_structured.json.gz` / `deepalgpro_test_ss3_structured.json.gz`
- `data/disorder/deepalgpro_train_disorder.json.gz` / `deepalgpro_test_disorder.json.gz`

Section C reads `results/rsa_regularization/sweep_summary.csv` produced by notebook 12.
On Colab this is read from Google Drive; run notebook 12 first and keep Drive mounted.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR = RUNTIME_ROOT / 'src'
PACKAGE_DIR = SRC_DIR / 'xallergen'
DATA_DIR = RUNTIME_ROOT / 'data'
SS3_DIR = DATA_DIR / 'ss3'
DISORDER_DIR = DATA_DIR / 'disorder'
RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [
    SRC_DIR, PACKAGE_DIR, DATA_DIR, SS3_DIR, DISORDER_DIR,
    RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'mtl_epitope_notebook_utils.py',
        'rsa_preprocessing.py',
    ]
    data_files = [
        'deepalgpro_train_cleaned.csv',
        'deepalgpro_test_cleaned.csv',
        'positives_splitB.csv',
    ]
    ss3_files = [
        'deepalgpro_train_ss3_structured.json.gz',
        'deepalgpro_test_ss3_structured.json.gz',
    ]
    disorder_files = [
        'deepalgpro_train_disorder.json.gz',
        'deepalgpro_test_disorder.json.gz',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.extend((f'{RAW}/data/ss3/{name}', SS3_DIR / name) for name in ss3_files)
    download_jobs.extend((f'{RAW}/data/disorder/{name}', DISORDER_DIR / name) for name in disorder_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'SS3 dir: {SS3_DIR}')
print(f'Disorder dir: {DISORDER_DIR}')
print(f'Output results dir: {RESULTS_DIR}')

# Section A — SS3-Structured Regularization Sweep

Tests whether penalising attention on coil/loop residues improves allergenicity classification
and residue-level localisation. The regularization loss is:

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls} + \lambda_\text{reg} \cdot \frac{1}{L}\sum_i \alpha_i (1 - f_i)$$

where $f_i = 1.0$ for helix (H) or strand (E) residues and $f_i = 0.0$ for coil/loop (C).
The $(1 - f_i)$ term is therefore 0 for structured residues and 1 for coil residues,
so the loss penalises attention weight on unstructured positions.

No new model heads or trainable components are introduced; the frozen ESM-2 backbone is unchanged.
Section C compares the best SS3-structured λ against the RSA baseline from notebook 12.

In [ ]:
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import (
    RANDOM_STATE,
    seed_everything,
)
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    inspect_rsa_inputs,
    load_rsa_lookup_dicts,
    run_lambda_rsa_sweep,
)
from xallergen.rsa_preprocessing import compute_rsa_ss3_structured_correlation

TRAIN_CSV = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
POSITIVES_SPLITB_CSV = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'
TRAIN_SS3_PATH = SS3_DIR / 'deepalgpro_train_ss3_structured.json.gz'
TEST_SS3_PATH = SS3_DIR / 'deepalgpro_test_ss3_structured.json.gz'
TRAIN_RSA_PATH = DATA_DIR / 'rsa' / 'deepalgpro_train_rsa.json.gz'
SS3_RESULTS_DIR = RESULTS_DIR / 'ss3_regularization'
SS3_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SS3_SWEEP_CSV = SS3_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    TRAIN_CSV, TEST_CSV, TRAIN_SS3_PATH, TEST_SS3_PATH,
    TRAIN_RSA_PATH, POSITIVES_SPLITB_CSV, BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

# λ=0 (unregularized baseline) is skipped — already present in the RSA sweep
# and identical across constraints. It appears in Section C for comparison.
SS3_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
    feature_key='ss3_structured',
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Feature: {SS3_CONFIG.feature_key}  (1.0=H/E structured, 0.0=C coil)')
print(f'λ values: {SS3_CONFIG.lambda_rsa_values}')
print(f'Output dir: {SS3_RESULTS_DIR}')

In [ ]:
ss3_train_df = pd.read_csv(TRAIN_CSV).copy()
ss3_test_df = pd.read_csv(TEST_CSV).copy()
for df in [ss3_train_df, ss3_test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence'] = df['sequence'].astype(str).str.strip().str.upper()
    df['label'] = df['label'].astype(int)

# rsa_min/rsa_max reflect the binary SS3-structured indicator (0.0 or 1.0)
display(inspect_rsa_inputs(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
))

In [ ]:
seed_everything(RANDOM_STATE)
ss3_train_split_df, ss3_val_split_df = train_test_split(
    ss3_train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=ss3_train_df['label'],
)
ss3_train_split_df = ss3_train_split_df.reset_index(drop=True).copy()
ss3_val_split_df = ss3_val_split_df.reset_index(drop=True).copy()

print(f'Train: {len(ss3_train_split_df):,} | Val: {len(ss3_val_split_df):,} | Test: {len(ss3_test_df):,}')

In [ ]:
ss3_train_lookup, ss3_test_lookup, ss3_lookup_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
    add_special_tokens=SS3_CONFIG.add_special_tokens,
)
display(pd.DataFrame([ss3_lookup_summary['train'], ss3_lookup_summary['test']]))

In [ ]:
import gzip, json

with gzip.open(TRAIN_RSA_PATH, 'rt') as f:
    rsa_train_lookup_raw = json.load(f)

# Pearson r between RSA and SS3-structured pooled over all train residues.
# Low |r| means the two constraints carry complementary signal.
compute_rsa_ss3_structured_correlation(rsa_train_lookup_raw, ss3_train_lookup)

In [ ]:
ss3_sweep_df = run_lambda_rsa_sweep(
    config=SS3_CONFIG,
    train_split_df=ss3_train_split_df,
    val_split_df=ss3_val_split_df,
    test_df=ss3_test_df,
    train_rsa_lookup=ss3_train_lookup,
    test_rsa_lookup=ss3_test_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=SS3_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
ss3_sweep_df.to_csv(SS3_SWEEP_CSV, index=False)
display(ss3_sweep_df)
print(f'Saved to: {SS3_SWEEP_CSV}')
torch.cuda.empty_cache()

In [ ]:
best_ss3 = ss3_sweep_df.loc[ss3_sweep_df['val_mcc'].idxmax()]
print('=== SS3-structured: best λ by val MCC ===')
print(f'  Best λ       : {best_ss3["lambda_rsa"]}')
print(f'  Val MCC      : {best_ss3["val_mcc"]:.4f}')
print(f'  Test MCC     : {best_ss3["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best_ss3["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best_ss3["residue_auroc"]:.4f}')

---
## Section B — Disorder Regularization Sweep

Tests whether penalising attention on ordered (low-disorder) residues improves allergenicity
classification. The same loss form as Section A, with per-residue disorder score as $f_i$:

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls} + \lambda_\text{reg} \cdot \frac{1}{L}\sum_i \alpha_i (1 - f_i)$$

where $f_i = 1.0$ for disordered residues and $f_i = 0.0$ for ordered residues.
The $(1 - f_i)$ term is 0 for disordered positions and 1 for ordered positions,
so the loss penalises attention on structured/ordered residues — reflecting that allergen
epitopes tend to localise in flexible, disordered regions.

In [ ]:
TRAIN_DISORDER_PATH = DISORDER_DIR / 'deepalgpro_train_disorder.json.gz'
TEST_DISORDER_PATH = DISORDER_DIR / 'deepalgpro_test_disorder.json.gz'
DISORDER_RESULTS_DIR = RESULTS_DIR / 'disorder_regularization'
DISORDER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DISORDER_SWEEP_CSV = DISORDER_RESULTS_DIR / 'sweep_summary.csv'

DISORDER_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
    feature_key='disorder',
)

for required_path in [TRAIN_DISORDER_PATH, TEST_DISORDER_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

seed_everything(RANDOM_STATE)
print(f'Feature: {DISORDER_CONFIG.feature_key}  (1.0=disordered, 0.0=ordered)')
print(f'λ values: {DISORDER_CONFIG.lambda_rsa_values}')
print(f'Output dir: {DISORDER_RESULTS_DIR}')

In [ ]:
display(inspect_rsa_inputs(
    train_rsa_path=TRAIN_DISORDER_PATH,
    test_rsa_path=TEST_DISORDER_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
))

In [ ]:
disorder_train_lookup, disorder_test_lookup, disorder_lookup_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_DISORDER_PATH,
    test_rsa_path=TEST_DISORDER_PATH,
    train_frame=ss3_train_df,
    test_frame=ss3_test_df,
    add_special_tokens=DISORDER_CONFIG.add_special_tokens,
)
display(pd.DataFrame([disorder_lookup_summary['train'], disorder_lookup_summary['test']]))

In [ ]:
disorder_sweep_df = run_lambda_rsa_sweep(
    config=DISORDER_CONFIG,
    train_split_df=ss3_train_split_df,
    val_split_df=ss3_val_split_df,
    test_df=ss3_test_df,
    train_rsa_lookup=disorder_train_lookup,
    test_rsa_lookup=disorder_test_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=DISORDER_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
disorder_sweep_df.to_csv(DISORDER_SWEEP_CSV, index=False)
display(disorder_sweep_df)
print(f'Saved to: {DISORDER_SWEEP_CSV}')
torch.cuda.empty_cache()

In [ ]:
best_disorder = disorder_sweep_df.loc[disorder_sweep_df['val_mcc'].idxmax()]
print('=== Disorder: best λ by val MCC ===')
print(f'  Best λ       : {best_disorder["lambda_rsa"]}')
print(f'  Val MCC      : {best_disorder["val_mcc"]:.4f}')
print(f'  Test MCC     : {best_disorder["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best_disorder["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best_disorder["residue_auroc"]:.4f}')

---
## Section C — Comparison Table

- **Baseline (λ=0)** — unregularized model, read from the RSA sweep's λ=0 entry.
- **RSA** — best λ > 0 by val MCC from notebook 12.
- **Disorder** — best λ > 0 by val MCC from Section B above.
- **SS3-structured** — best λ > 0 by val MCC from Section A above.

In [ ]:
RSA_SWEEP_CSV = RESULTS_DIR / 'rsa_regularization' / 'sweep_summary.csv'
if not RSA_SWEEP_CSV.exists():
    raise FileNotFoundError(
        f'RSA sweep summary not found at {RSA_SWEEP_CSV}. '
        'Run notebook 12 first and ensure its output CSV is present in the results directory.'
    )

def _row(csv_path, label, select_best=True):
    df = pd.read_csv(csv_path)
    row = df.loc[df['val_mcc'].idxmax()] if select_best else df.loc[df['lambda_rsa'].eq(0.0)].iloc[0]
    return {
        'Constraint': label,
        'Best λ': row['lambda_rsa'],
        'Val MCC': round(float(row['val_mcc']), 4),
        'Test MCC': round(float(row['test_mcc']), 4),
        'Test AUROC': round(float(row['test_auroc']), 4),
        'Residue AUROC': round(float(row['residue_auroc']), 4),
    }

comparison_df = pd.DataFrame([
    _row(RSA_SWEEP_CSV,      'Baseline (λ=0)',  select_best=False),
    _row(RSA_SWEEP_CSV,      'RSA',             select_best=True),
    _row(DISORDER_SWEEP_CSV, 'Disorder',        select_best=True),
    _row(SS3_SWEEP_CSV,      'SS3-structured',  select_best=True),
]).set_index('Constraint')

print('=== Comparison (λ>0 rows selected by val MCC) ===')
display(comparison_df)

In [ ]:
# Shut down the runtime after everything above has finished.
os.kill(os.getpid(), 9)

: 